<a href="https://colab.research.google.com/github/Jubaida-78/Machine-Learning-with-python/blob/main/Litchi_Sensor_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

LOAD CSV

In [ ]:
df=pd.read_csv('/content/Litchi Sensor Data.csv')
df.head()

CHECK DATASET

In [ ]:
print(df.shape)
print(df.columns)
print(df.info())

CHECK TARGET CLASS/CATEGOY

In [ ]:
print(df['Category'].value_counts())

#Correlation heatmap

In [ ]:
print(df.describe())
df.drop('ImageID', axis=1).hist(figsize=(12,8), bins=30)
plt.tight_layout()
plt.show()
sns.heatmap(df.drop(['ImageID','Category'],axis=1).corr(),annot=True,cmap='coolwarm')
plt.show()

ENCODE LABELS

In [ ]:
encoder=LabelEncoder()
df['Category']=encoder.fit_transform(df.Category)
print(df.head())

In [ ]:
print(df.tail())

SELECT FEATURES

In [ ]:
X=df[['Moisture','Firmness','pH','TSS','Ethanol','CO2','Formaldehyde']]
y=df['Category']

NORMALIZE DATA:
Normalization in CNN means scaling image pixel values (0–255) into a smaller range like 0–1 to improve training performance.


In [ ]:
scaler=StandardScaler()
X=scaler.fit_transform(X)

TRAIN TEST SPLIT

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)

RESHAPE FOR CNN

In [ ]:
X_train=X_train.reshape(X_train.shape[0],X_train.shape[1],1)
X_test=X_test.reshape(X_test.shape[0],X_test.shape[1],1)

BUILD CNN MODEL

In [ ]:
cnn_model=tf.keras.Sequential([
    tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        input_shape=(7,1)
    ),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(
        filters=128,
        kernel_size=2,
        activation='relu'
    ),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

COMPILE MODEL(evaluation metrics before training)

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

TRAIN MODEL

In [ ]:
history=cnn_model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

EVALUATE MODEL

In [ ]:
loss,accuracy=cnn_model.evaluate(X_test,y_test)
print('CNN Accuracy:',accuracy)

ACCURACY GRAPH

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train','Validation'])
plt.show()

CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
y_pred=cnn_model.predict(X_test)
y_pred=(y_pred>0.5)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

MOVING TO PRETRAINED/ADVANCED MODEL

In [ ]:
!pip install tsai

In [ ]:
from tsai.all import *

PREPARING DATA FOR InceptionTime

In [ ]:
X_train_tsai=np.transpose(X_train,(0,2,1))

X_test_tsai=np.transpose(X_test,(0,2,1))

CREATING DATALOADERS

In [ ]:
X=np.concatenate((X_train_tsai,X_test_tsai))
y=np.concatenate((y_train,y_test))

In [ ]:
splits=get_splits(y,valid_size=0.2,random_state=42)

In [ ]:
dls=get_ts_dls(
    X,
    y,
    splits=splits,
    bs=32
)

BUILDING InceptionTime MODEL

In [ ]:
learn=ts_learner(
    dls,
    InceptionTime,
    metrics=accuracy
)

TRAIN InceptionTime

In [ ]:
learn.fit_one_cycle(30,1e-3)

EVALUATE MODEL

In [ ]:
results=learn.validate()
print(results)

PREDICTIONS

In [ ]:
preds,targets=learn.get_preds()
pred_labels=preds.argmax(dim=1)

CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
cm=confusion_matrix(targets,pred_labels)
print(cm)
print(classification_report(targets,pred_labels))

SAVE MODEL

In [ ]:
learn.export('inceptiontime_model.pkl')


In [ ]:
cnn_model.save('cnn_model.h5')